# Workspace Coverage Analysis (thesis-grade)

Loads the most recent `workspace_coverage` run and reports ISO 9283-flavoured metrics across the workspace, plus an application-zoned spatial map. Volcaniarm is 2-DOF planar; metrics are in the Y-Z plane (X dropped).

**Per-goal metrics (computed below):**
- **`RP_yz`** per goal — local repeatability at each pose. Frame-independent.
- **`AP_yz`** per goal — local accuracy (centroid vs commanded). Biased by URDF tag-mount offset; per-goal *variation* of `AP_yz` is the diagnostic for URDF-mount accuracy.
- **Worst-case** sample-to-centroid distance per goal — agricultural-targeting tail metric.

**Across-workspace summaries:**
- Overall `RP_yz` (worst per-goal repeatability) and `AP_yz` (worst per-goal accuracy).
- Spatial heatmap of `dr_yz` magnitude with **application-threshold zones** overlaid (acceptable / marginal / failing for weeding).
- Distribution of `d_diff` (`|d_tag| − |d_fk|`) across goals: small spread = URDF mounts consistent across the workspace.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from volcaniarm_calibration.analysis import (
    accuracy_iso9283, align_fk_to_tag, latest_run, list_runs, load_run,
    repeatability_iso9283, summary, threshold_color, threshold_zone,
    WEEDING_ACCEPTABLE_MM, WEEDING_MARGINAL_MM,
)

TEST_NAME = 'workspace_coverage'
RUN_DIR = latest_run(TEST_NAME)
run = load_run(RUN_DIR)
config, tag, fk = run['config'], run['tag'], run['fk']
print(f'run_id:             {config.get("run_id")}')
print(f'status:             {config.get("status")}')
print(f'goals:              {len(config["goals"])}')
print(f'samples per visit:  {config["samples_per_visit"]}')
print(f'tag rows: {len(tag)}, fk rows: {len(fk)}')

target = tag[tag['phase'] == 'target'].copy()
target['target_idx'] = target['target_idx'].astype(int)

## Per-goal repeatability and accuracy

For each goal, compute `RP_yz` (local repeatability), `AP_yz` (local accuracy, biased), and worst-case sample-to-centroid distance. Sorted by `RP_yz` so problem goals surface at the top.

In [ ]:
rows = []
for ti in sorted(target['target_idx'].unique()):
    sub = target[target['target_idx'] == ti]
    rep = repeatability_iso9283(sub, dims=('y', 'z'))
    gy, gz = config['goals'][ti - 1]
    ap = accuracy_iso9283(sub, commanded=(gy, gz), dims=('y', 'z'))
    rows.append({
        'goal': ti,
        'goal_y': gy,
        'goal_z': gz,
        'n': rep['n'],
        'RP_mm': rep['RP_m'] * 1000,
        'worst_mm': rep['worst_m'] * 1000,
        'AP_mm': ap['AP_m'] * 1000,
        'offset_y_mm': ap['offset'][0] * 1000,
        'offset_z_mm': ap['offset'][1] * 1000,
        'zone_RP': threshold_zone(rep['RP_m'] * 1000),
        'zone_AP': threshold_zone(ap['AP_m'] * 1000),
    })
per_goal = pd.DataFrame(rows).sort_values('RP_mm', ascending=False)
print(per_goal.to_string(index=False))

## Workspace-level summary (worst-case across goals)

Headline thesis numbers: the worst-case `RP` and `AP` over the entire tested workspace. For weeding, you want the **worst-case** ≤ acceptable threshold.

In [ ]:
rp_stats = summary(per_goal['RP_mm'] / 1000)
ap_stats = summary(per_goal['AP_mm'] / 1000)
worst_stats = summary(per_goal['worst_mm'] / 1000)
print(f'RP_yz across goals:    {rp_stats.in_mm()}')
print(f'AP_yz across goals:    {ap_stats.in_mm()}')
print(f'worst-case d_to_cent:  {worst_stats.in_mm()}')
print(f'\nworst-case RP_yz: {per_goal["RP_mm"].max():.3f} mm   '
      f'(zone: {threshold_zone(per_goal["RP_mm"].max())})')
print(f'worst-case AP_yz: {per_goal["AP_mm"].max():.3f} mm   '
      f'(zone: {threshold_zone(per_goal["AP_mm"].max())})')

## URDF-mount-bias diagnostic (across-goal `d_diff` spread)

If the URDF apriltag mount values are right, `d_diff = |d_tag| - |d_fk|` should be **constant across goals**. Variation in `d_diff` is a signature that the URDF apriltag offset doesn't match physical reality, with magnitude indicating how off.

In [ ]:
target['d_tag_m'] = np.sqrt(target['x']**2 + target['y']**2 + target['z']**2)
fk_dist = fk.copy()
fk_dist['d_fk_m'] = np.sqrt(fk_dist['fk_x']**2 + fk_dist['fk_y']**2 + fk_dist['fk_z']**2)
fk_dist['target_idx'] = fk_dist['target_idx'].astype(int)
merged_d = target.merge(fk_dist[['cycle', 'target_idx', 'd_fk_m']],
                        on=['cycle', 'target_idx'], how='left')
merged_d['d_diff_m'] = merged_d['d_tag_m'] - merged_d['d_fk_m']
by_goal_diff = merged_d.groupby('target_idx')['d_diff_m'].agg(
    mean='mean', std='std', n='count').reset_index()
by_goal_diff['mean_mm'] = by_goal_diff['mean'] * 1000
by_goal_diff['std_mm'] = by_goal_diff['std'] * 1000
print(by_goal_diff.to_string(index=False))
diff_summary = summary(by_goal_diff['mean'])
print(f'\nd_diff across goals:   {diff_summary.in_mm()}')
print(f'  spread of d_diff:    {diff_summary.std*1000:.3f} mm')
print(f'  small spread (< few mm) = URDF mounts are consistent across workspace.')

## Top-down workspace scatter with thresholds

Each goal annotated with its `RP_yz` and zone-coloured. Visual reading: green = acceptable for weeding, yellow = marginal, red = failing.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 8))

goals = np.array(config['goals'])
ax.scatter(goals[:, 0], goals[:, 1], marker='*', s=300, c='red',
           label='goals', zorder=5)
ax.scatter(fk['fk_y'], fk['fk_z'], marker='o', s=120, facecolors='none',
           edgecolors='orange', linewidths=2, label='FK', zorder=4)
ax.scatter(target['y'], target['z'], marker='x', s=60, c='steelblue',
           alpha=0.5, label='tag samples', zorder=3)

for _, row in per_goal.iterrows():
    ti = int(row['goal'])
    rp_mm = row['RP_mm']
    color = threshold_color(rp_mm)
    ax.annotate(f'g{ti}\nRP={rp_mm:.1f}mm',
                (row['goal_y'], row['goal_z']),
                textcoords='offset points', xytext=(10, 10),
                fontsize=8, color=color,
                bbox=dict(boxstyle='round,pad=0.2', edgecolor=color,
                          facecolor='white', alpha=0.8))

ax.set_xlabel('y [m]')
ax.set_ylabel('z [m]')
ax.set_title(f'workspace coverage  ({config["run_id"]})  '
             f'green=acceptable, yellow=marginal, red=failing')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.legend(loc='best', fontsize=8)
fig.tight_layout()

## Spatial heatmap of RP_yz (or AP_yz)

Interpolated colourmap of repeatability (or accuracy) across the workspace. Darker red = worse. Useful for identifying parts of the workspace that fail for weeding.

In [ ]:
if len(per_goal) >= 3:
    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    for ax, metric_col, title in (
        (axes[0], 'RP_mm', 'RP_yz [mm] (repeatability)'),
        (axes[1], 'AP_mm', 'AP_yz [mm] (accuracy, biased)'),
    ):
        pg = per_goal.sort_values('goal')
        ys = pg['goal_y'].to_numpy()
        zs = pg['goal_z'].to_numpy()
        vals = pg[metric_col].to_numpy()
        # Discrete contour at threshold values for clear zone reading
        levels = [0, WEEDING_ACCEPTABLE_MM, WEEDING_MARGINAL_MM,
                  max(vals.max(), WEEDING_MARGINAL_MM + 1)]
        colors = ['#2e9c4a', '#c79a3a', '#d04b4b']
        try:
            tcf = ax.tricontourf(ys, zs, vals, levels=levels, colors=colors,
                                 alpha=0.5)
        except Exception:
            # Fall back to a smooth colormap if triangulation issues
            tcf = ax.tricontourf(ys, zs, vals, levels=12, cmap='RdYlGn_r')
        ax.scatter(ys, zs, c='black', s=30, zorder=5)
        for ti, y, z, v in zip(pg['goal'].values, ys, zs, vals):
            ax.annotate(f'g{int(ti)}\n{v:.1f}', (y, z),
                        textcoords='offset points', xytext=(6, 6),
                        fontsize=7)
        cbar = plt.colorbar(tcf, ax=ax)
        cbar.set_label(metric_col)
        ax.set_xlabel('y [m]')
        ax.set_ylabel('z [m]')
        ax.set_title(title)
        ax.set_aspect('equal')
    fig.tight_layout()
else:
    print(f'only {len(per_goal)} goals; need at least 3 for the heatmap')

## Thesis caveats summary

- `RP_yz` is reportable as-is — frame-independent, no URDF bias.
- `AP_yz` carries the URDF apriltag mount offset bias. Per-goal *variation* of `AP_yz` is the diagnostic for URDF-mount accuracy across the workspace; a small spread (mm-scale) means the URDF is consistent.
- The threshold zones (`WEEDING_ACCEPTABLE_MM`, `WEEDING_MARGINAL_MM`) come from `analysis/metrics.py`; tune for your specific weed-targeting precision needs.